# Wilma — Exploratory Data Analysis

Look at the training data before we train models.
Run cells top-to-bottom with Shift+Enter.

## 1. Load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data" / "processed" if Path.cwd().name == "notebooks" else Path("data") / "processed"

train = pd.read_csv(DATA_DIR / "wilma_v1_train.csv")
val   = pd.read_csv(DATA_DIR / "wilma_v1_val.csv")
test  = pd.read_csv(DATA_DIR / "wilma_v1_test.csv")

print(f"train: {len(train):>6} rows")
print(f"val:   {len(val):>6} rows")
print(f"test:  {len(test):>6} rows")
train.head()

## 2. Class balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train["label"].value_counts().plot.barh(ax=axes[0], title="Binary labels")
train["category"].value_counts().plot.barh(ax=axes[1], title="Categories")
plt.tight_layout()
plt.show()
print((train["label"].value_counts(normalize=True) * 100).round(2))

## 3. Where the data comes from

In [ ]:
train["source"].value_counts().plot.barh(figsize=(10, 3), title="Source distribution")
plt.tight_layout()
plt.show()
print(train.groupby("source")["label"].value_counts().unstack(fill_value=0))

## 4. Message length

In [ ]:
train["char_len"].clip(upper=2000).hist(bins=60, figsize=(10, 4))
plt.title("Message length (clipped at 2000 chars)")
plt.xlabel("chars")
plt.show()
print(train.groupby("label")["char_len"].describe().astype(int))

## 5. Languages

In [ ]:
print(train["language"].value_counts().head(10))
print(f"\n% English: {(train['language']=='en').mean()*100:.1f}%")

## 6. Words that scream 'scam'

Which words appear way more in scams than in legitimate messages?

In [ ]:
import re

def tokenize(text):
    return re.findall(r"[a-zA-Z<>]+", str(text).lower())

scam_counts, legit_counts = Counter(), Counter()
for _, row in train.iterrows():
    bag = scam_counts if row["label"] == "scam" else legit_counts
    bag.update(tokenize(row["text"]))

total_scam = sum(scam_counts.values())
total_legit = sum(legit_counts.values())
scores = []
for w in set(scam_counts) | set(legit_counts):
    if scam_counts[w] + legit_counts[w] >= 50:
        s = scam_counts[w] / total_scam
        l = legit_counts[w] / total_legit
        scores.append((w, np.log((s + 1e-10) / (l + 1e-10)), scam_counts[w], legit_counts[w]))

scores.sort(key=lambda x: x[1], reverse=True)
print("Top 20 scam-leaning words:")
for w, score, sc, lc in scores[:20]:
    print(f"  {w:20s}  log-odds={score:+.2f}   scam={sc:>6}  legit={lc:>6}")

## 7. Data leakage check

Make sure no rows leak between train/val/test.

In [ ]:
train_h, val_h, test_h = set(train["raw_hash"]), set(val["raw_hash"]), set(test["raw_hash"])
print(f"train ∩ val:  {len(train_h & val_h)}")
print(f"train ∩ test: {len(train_h & test_h)}")
print(f"val ∩ test:   {len(val_h & test_h)}")
print("\nAll zeros = no leakage")